In [1]:
import pandas as pd
from pathlib import Path

In [2]:
PILOT_DIR = Path("../data/pilot")            
POOL_PATH = Path("../data/processed/pool.csv")

PILOT_FILES = {
    "qwen":    "pilot_Qwen3-8B.csv",
    "mistral": "pilot_Mistral-7B-Instruct-v0.3.csv",
    "llama":   "pilot_Llama-3.1-8B-Instruct.csv",
    "gemma":   "pilot_gemma-2-9b-it.csv",
}

MAIN_MODELS = ["qwen", "gemma", "llama"]

In [3]:
def load_pilots(files):
    """one row per sentence: each model's label and confidence, plus the human label"""
    merged = None
    for name, fname in files.items():
        path = PILOT_DIR / fname
        if not path.exists():
            print(f"skipping {name}: {fname} not found")
            continue

        df = pd.read_csv(path)
        df = df[["item_id", "human_label", "llm_label", "confidence"]]
        df = df.rename(columns={"llm_label": f"{name}_label", "confidence": f"{name}_conf"})

        if merged is None:
            merged = df
        else:
            merged = merged.merge(df.drop(columns="human_label"), on="item_id")
        print(f"loaded {name}")
    return merged


def has_models(df, models):
    """True if every model in the list was loaded"""
    return all(f"{m}_label" in df.columns for m in models)


def add_agreement(df, models):
    """add agreement columns for the chosen models"""
    label_cols = [f"{m}_label" for m in models]
    conf_cols = [f"{m}_conf" for m in models]

    df = df.copy()
    df["n_different"] = df[label_cols].nunique(axis=1)          # 1 = all agree
    df["all_agree"] = df["n_different"] == 1
    df["no_majority"] = df["n_different"] == len(models)        # every model said something different
    df["majority_label"] = df[label_cols].mode(axis=1)[0]
    df["majority_correct"] = df["majority_label"] == df["human_label"]
    df["mean_conf"] = df[conf_cols].mean(axis=1)
    return df


def agreement_report(df, models):
    """explain the agreement results in plain sentences"""
    n = len(df)
    agree = df[df["all_agree"]]
    kept = len(agree)
    kept_right = agree["majority_correct"].sum()
    kept_wrong = kept - kept_right

    print(f"=== {' + '.join(models)} ({n} sentences) ===\n")

    print("If we only keep labels the models agree on:")
    print(f"  keep  {kept:3} sentences ({kept / n:.0%})  - all models gave the same label")
    print(f"  drop  {n - kept:3} sentences ({(n - kept) / n:.0%})  - the models disagreed\n")

    print(f"Of the {kept} labels we keep:")
    print(f"  {kept_right:3} are correct ({kept_right / kept:.0%})")
    print(f"  {kept_wrong:3} are wrong   ({kept_wrong / kept:.0%})  <- agreement did not catch these\n")

    print("How often each model is right on its own:")
    for m in models:
        print(f"  {m:9} {(df[f'{m}_label'] == df['human_label']).mean():.0%}")
    if len(models) >= 3:
        print(f"  {'majority':9} {df['majority_correct'].mean():.0%}  (most common answer of the 3)")

    conf_right = agree.loc[agree["majority_correct"], "mean_conf"].mean()
    conf_wrong = agree.loc[~agree["majority_correct"], "mean_conf"].mean()
    print("\nAre the models less sure when they agree but are wrong?")
    print(f"  agreed and correct: {conf_right:.1%} average confidence")
    print(f"  agreed but wrong:   {conf_wrong:.1%} average confidence")
    if conf_wrong < conf_right:
        print(f"  -> yes, {100 * (conf_right - conf_wrong):.1f} points lower")
    else:
        print("  -> no, they are just as confident")


def compare_model_sets(pilots, model_sets):
    """one row per combination of models, for side-by-side comparison"""
    rows = []
    for models in model_sets:
        if not has_models(pilots, models):
            print(f"skipping {' + '.join(models)}: not all models loaded")
            continue

        df = add_agreement(pilots, models)
        agree = df[df["all_agree"]]
        rows.append({
            "models": " + ".join(models),
            "labels kept": len(agree),
            "% of kept that are correct": f"{agree['majority_correct'].mean():.0%}",
            "kept but wrong": (~agree["majority_correct"]).sum(),
            "confidence when correct": f"{agree.loc[agree['majority_correct'], 'mean_conf'].mean():.1%}",
            "confidence when wrong": f"{agree.loc[~agree['majority_correct'], 'mean_conf'].mean():.1%}",
        })
    return pd.DataFrame(rows)

In [4]:
pilots = load_pilots(PILOT_FILES)

loaded qwen
loaded mistral
loaded llama
loaded gemma


In [5]:
main = add_agreement(pilots, MAIN_MODELS)
agreement_report(main, MAIN_MODELS)

=== qwen + gemma + llama (100 sentences) ===

If we only keep labels the models agree on:
  keep   71 sentences (71%)  - all models gave the same label
  drop   29 sentences (29%)  - the models disagreed

Of the 71 labels we keep:
   63 are correct (89%)
    8 are wrong   (11%)  <- agreement did not catch these

How often each model is right on its own:
  qwen      79%
  gemma     74%
  llama     83%
  majority  81%  (most common answer of the 3)

Are the models less sure when they agree but are wrong?
  agreed and correct: 97.7% average confidence
  agreed but wrong:   95.3% average confidence
  -> yes, 2.4 points lower


In [9]:
pool = pd.read_csv(POOL_PATH)

agreed_wrong = main[main["all_agree"] & ~main["majority_correct"]]
agreed_wrong = agreed_wrong.merge(pool[["item_id", "sentence"]], on="item_id")
print(agreed_wrong[["sentence", "human_label", "majority_label", "mean_conf"]])

                                            sentence human_label  \
0  The Russian gas giant invested another 46 mill...     neutral   
1  Operating income was EUR -24.9 mn , including ...     neutral   
2  Kaupthing forecasts that Finnish-Swedish Stora...     neutral   
3  `` Our design team has long admired Marimekko ...    positive   
4  Upon completion , Keskuskatu will form the bac...    positive   
5  In complying with the European Water Framework...    positive   
6  Jobs will not be transferred from Finland , ho...    positive   
7  TVO 's two-unit 1,740 MW Olkiluoto plant gener...     neutral   

  majority_label  mean_conf  
0       positive   0.861887  
1       negative   0.996498  
2       negative   0.982317  
3        neutral   0.989634  
4        neutral   0.924251  
5        neutral   0.974719  
6        neutral   0.992277  
7       positive   0.902746  


In [10]:
print(compare_model_sets(pilots, [
    ["qwen", "mistral", "llama"],
    ["qwen", "llama"],
    ["qwen", "gemma", "llama"],
]))

                   models  labels kept % of kept that are correct  \
0  qwen + mistral + llama           50                        88%   
1            qwen + llama           84                        87%   
2    qwen + gemma + llama           71                        89%   

   kept but wrong confidence when correct confidence when wrong  
0               6                   97.4%                 93.1%  
1              11                   96.5%                 92.3%  
2               8                   97.7%                 95.3%  
